In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import re
import warnings
from typing import List

# Suppress warnings
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# ==============================================================================
# USER CONFIGURABLE PARAMETERS
# ==============================================================================

# --- File Paths and Patterns ---
# Directory containing the Excel files generated by Code 1.
# Updated to match Script 1 output directory for consistency.
SEARCH_DIR = "."

# Pattern to match the Excel filenames.
FILENAME_PATTERN = "pairwise_distance_repl_*.xlsx"

# Name of the final output Excel file.
OUTPUT_FILENAME = "pairwise_consensus.xlsx"

# The sheet name to read from the input files (must match Code 1).
INPUT_SHEET_NAME = 'filtered_data'

# --- Thresholds ---
# The minimum occupancy threshold (0.5 means > 50% occupancy).
OCCUPANCY_THRESHOLD = 0.5

# The percentage tolerance for the kinetic ratio-based classification (0.25 means +/- 25%).
RATIO_THRESHOLD = 0.25

# --- Column Definitions ---
# Columns that uniquely identify an interaction pair (Must match Code 1).
PAIR_COLS = [
    'chain', 'residue_pdb_resseq', 'resname',
    'chain_partner', 'partner_residue_pdb_resseq', 'partner_resname'
]

# --- Kinetic Window Configuration (CRITICAL) ---
# [WARNING] Code 1 generates window names dynamically based on frame counts (e.g., "occ_1_200").
# Option A (Recommended): Leave this list EMPTY []. The code will auto-detect the columns.
# Option B (Manual): List the EXACT column names from your Code 1 output files if you want strict control.
# Example: MANUAL_WINDOW_COLS = ['occ_1_3333', 'occ_3334_6666', 'occ_6667_10000']
MANUAL_WINDOW_COLS = [] 

# ==============================================================================
# CORE LOGIC
# ==============================================================================

def extract_repl_number(filename: Path) -> int:
    """Extracts the replication number ('repl') from the filename string."""
    match = re.search(r'repl_(\d+)', str(filename.name))
    if match:
        return int(match.group(1))
    return -1

def get_window_columns(df_columns: List[str]) -> List[str]:
    """
    Determines which columns are time windows based on User Config or Auto-detection.
    """
    if MANUAL_WINDOW_COLS:
        # Validate that the user-specified columns actually exist in the file
        missing = [c for c in MANUAL_WINDOW_COLS if c not in df_columns]
        if missing:
            print(f"Global WARNING ⚠️ - User-specified columns missing in file: {missing}")
            return []
        return MANUAL_WINDOW_COLS
    
    # Auto-detection: Look for columns starting with 'occ_' that contain numbers
    # and exclude 'occupancy_total' and 'occ_last2' (if it exists)
    detected = [c for c in df_columns if c.startswith('occ_') and c != 'occupancy_total' and c != 'occ_last2']
    
    # Sort them by the starting frame number to ensure time order (occ_1_... before occ_200_...)
    def get_start_frame(col_name):
        try:
            # Extract the first number after 'occ_'
            parts = col_name.split('_')
            return int(parts[1])
        except (IndexError, ValueError):
            return 999999
            
    return sorted(detected, key=get_start_frame)

def process_single_file(filepath: Path, repl_number: int):
    """
    Reads the filtered sheet from a single Excel file.
    """
    filename = filepath.name
    try:
        df = pd.read_excel(filepath, sheet_name=INPUT_SHEET_NAME)
    except Exception as e:
        print(f"File [{filename}] ERROR ❌ - Could not read sheet '{INPUT_SHEET_NAME}': {e}")
        return None

    if df.empty:
        print(f"File [{filename}] INFO - Sheet '{INPUT_SHEET_NAME}' is empty.")
        return None

    # Identify window columns using the helper function
    window_cols = get_window_columns(df.columns)
    
    # Ensure critical columns exist
    missing_cols = [col for col in PAIR_COLS if col not in df.columns]
    if missing_cols:
        print(f"File [{filename}] ERROR ❌ - Missing identification columns: {missing_cols}. Skipping.")
        return None

    # Calculate average of the last two windows for kinetic analysis
    if len(window_cols) >= 2:
        last_two = window_cols[-2:]
        df['occ_last2'] = df[last_two].mean(axis=1)
    else:
        # Fallback if less than 2 windows exist
        df['occ_last2'] = df['occupancy_total']

    df['repl'] = repl_number

    # Select columns to keep
    cols_to_keep = ['repl'] + PAIR_COLS + ['mean_distance_nm', 'median_distance_nm', 'occupancy_total'] + window_cols + ['occ_last2']
    
    # Filter to only existing columns
    cols_to_keep = [c for c in cols_to_keep if c in df.columns]
    
    df_processed = df[cols_to_keep].copy()
    
    print(f"Replica [{repl_number}] INFO - Loaded {len(df_processed)} interactions. Windows used: {len(window_cols)}")
    return df_processed

def calculate_overall_stability(all_data_df, total_replicas):
    """
    Calculates the consensus score based on the number of replicas.
    """
    print("\nGlobal INFO - Starting overall stability calculation (Consensus Score)...")

    if all_data_df.empty:
        return pd.DataFrame()

    df_filtered_score = all_data_df[all_data_df['occupancy_total'] >= OCCUPANCY_THRESHOLD].copy()

    if df_filtered_score.empty:
        print("Global INFO - No pairs meet the occupancy criteria for scoring.")
        return pd.DataFrame()

    agg_df = df_filtered_score.groupby(PAIR_COLS).agg(
        score=('repl', 'nunique'),
        replicas=('repl', lambda x: ','.join(map(str, sorted(x.unique()))))
    ).reset_index()

    half_max_score = total_replicas / 2.0

    def determine_consensus_level(score):
        if score == total_replicas:
            return 'PRIMARY (essential)'
        elif score >= half_max_score:
            return 'SECONDARY (auxiliary)'
        elif score >= 1:
            return 'TERTIARY (accessory)'
        return 'N/A'

    agg_df['consensus_level'] = agg_df['score'].apply(determine_consensus_level)
    agg_df.sort_values(by=['score', 'chain', 'residue_pdb_resseq'], ascending=[False, True, True], inplace=True)

    print(f"Global INFO - Stability calculated for {len(agg_df)} unique pairs.")
    return agg_df

def create_late_stage_emergence(all_data_df, ratio_threshold):
    """
    Analyzes kinetic trends based on window ratios.
    """
    print("\nGlobal INFO - Starting late-stage emergence calculation...")

    if all_data_df.empty:
        return pd.DataFrame()

    df_filtered = all_data_df[all_data_df['occ_last2'] >= OCCUPANCY_THRESHOLD].copy()

    if df_filtered.empty:
        return pd.DataFrame()

    # Determine which columns to use for kinetics
    window_cols = get_window_columns(df_filtered.columns)

    if len(window_cols) < 3:
        print(f"Global WARNING - Only {len(window_cols)} windows found. Kinetic analysis requires at least 3 windows (Start, Mid, End). Skipping.")
        return df_filtered

    # Select the first 3 windows found (or specific ones if you want to hardcode index 0, 1, 2)
    occ_1_col = window_cols[0]
    occ_2_col = window_cols[1]
    occ_3_col = window_cols[2]

    # Avoid division by zero
    epsilon = 1e-6
    occ_1_safe = df_filtered[occ_1_col].replace(0, epsilon)
    occ_2_safe = df_filtered[occ_2_col].replace(0, epsilon)

    df_filtered['ratio_1_2'] = df_filtered[occ_2_col] / occ_1_safe
    df_filtered['ratio_2_3'] = df_filtered[occ_3_col] / occ_2_safe

    T = ratio_threshold

    # Classify Transition 1 -> 2
    df_filtered['trans_1_2'] = 'Stable'
    df_filtered.loc[df_filtered['ratio_1_2'] >= (1 + T), 'trans_1_2'] = 'Increase'
    df_filtered.loc[df_filtered['ratio_1_2'] <= (1 - T), 'trans_1_2'] = 'Decrease'

    # Classify Transition 2 -> 3
    df_filtered['trans_2_3'] = 'Stable'
    df_filtered.loc[df_filtered['ratio_2_3'] >= (1 + T), 'trans_2_3'] = 'Increase'
    df_filtered.loc[df_filtered['ratio_2_3'] <= (1 - T), 'trans_2_3'] = 'Decrease'

    transition_map = {
        ('Stable', 'Stable'): 'consistently stable',
        ('Stable', 'Increase'): 'late emerging',
        ('Stable', 'Decrease'): 'late fading',
        ('Increase', 'Stable'): 'stabilizing',
        ('Increase', 'Increase'): 'emerging',
        ('Increase', 'Decrease'): 'transient peak',
        ('Decrease', 'Stable'): 'early fading',
        ('Decrease', 'Increase'): 'transient trough',
        ('Decrease', 'Decrease'): 'destabilizing',
    }

    def get_category(row):
        return transition_map.get((row['trans_1_2'], row['trans_2_3']), 'Unclassified')

    df_filtered['kinetic_category'] = df_filtered.apply(get_category, axis=1)

    cols = ['repl'] + PAIR_COLS + [occ_1_col, occ_2_col, occ_3_col, 'occ_last2', 'ratio_1_2', 'ratio_2_3', 'kinetic_category']
    cols = [c for c in cols if c in df_filtered.columns]
    
    return df_filtered[cols]

def create_category_dictionary(total_replicas, ratio_threshold):
    half_max = total_replicas / 2
    data = [
        ['CONSENSUS LEVELS', '', ''],
        ['PRIMARY (essential)', f'Score = {total_replicas}', 'Present in ALL replicas.'],
        ['SECONDARY (auxiliary)', f'Score >= {half_max}', 'Present in at least HALF the replicas.'],
        ['TERTIARY (accessory)', f'Score >= 1', 'Present in at least one replica.'],
        ['', '', ''],
        ['KINETIC CATEGORIES', f'Tolerance = +/- {ratio_threshold*100}%', ''],
        ['consistently stable', 'Stable -> Stable', 'Steady occupancy.'],
        ['late emerging', 'Stable -> Increase', 'Appears later.'],
        ['late fading', 'Stable -> Decrease', 'Disappears later.'],
        ['stabilizing', 'Increase -> Stable', 'Grows early, then holds.'],
        ['emerging', 'Increase -> Increase', 'Continuous growth.'],
        ['transient peak', 'Increase -> Decrease', 'Peaks in middle.'],
        ['early fading', 'Decrease -> Stable', 'Drops early.'],
        ['destabilizing', 'Decrease -> Decrease', 'Continuous loss.']
    ]
    return pd.DataFrame(data, columns=['Category', 'Criterion', 'Definition'])

def run_consensus_analysis():
    search_path = Path(SEARCH_DIR)
    
    if not search_path.exists():
        print(f"Global ERROR ❌ - Directory not found: {SEARCH_DIR}")
        return

    print(f"Global INFO - Searching for files matching '{FILENAME_PATTERN}' in {SEARCH_DIR}...")
    files = list(search_path.glob(FILENAME_PATTERN))
    
    valid_files = []
    for f in files:
        r_num = extract_repl_number(f)
        if r_num != -1:
            valid_files.append((r_num, f))
    
    valid_files.sort(key=lambda x: x[0])
    
    if not valid_files:
        print("Global ERROR ❌ - No matching files found.")
        return

    print(f"Global INFO - Found {len(valid_files)} valid replica files.")

    all_dfs = []
    for repl_num, filepath in valid_files:
        df = process_single_file(filepath, repl_num)
        if df is not None:
            all_dfs.append(df)

    if not all_dfs:
        print("Global ERROR ❌ - No data loaded.")
        return

    master_df = pd.concat(all_dfs, ignore_index=True)
    total_replicas_found = len(valid_files)

    df_stability = calculate_overall_stability(master_df, total_replicas_found)
    df_kinetics = create_late_stage_emergence(master_df, RATIO_THRESHOLD)
    df_dictionary = create_category_dictionary(total_replicas_found, RATIO_THRESHOLD)

    output_path = search_path / OUTPUT_FILENAME
    
    try:
        with pd.ExcelWriter(output_path, engine='xlsxwriter') as writer:
            if not df_stability.empty:
                df_stability.to_excel(writer, sheet_name='consensus_stability', index=False)
            if not df_kinetics.empty:
                df_kinetics.to_excel(writer, sheet_name='kinetics_analysis', index=False)
            
            master_df.to_excel(writer, sheet_name='concat_all_data', index=False)
            df_dictionary.to_excel(writer, sheet_name='definitions', index=False)
            
        print(f"\nGlobal SUCCESS ✅ - Consensus analysis complete. Output: {output_path}")

    except Exception as e:
        print(f"Global ERROR ❌ - Failed to write output Excel: {e}")

if __name__ == "__main__":
    run_consensus_analysis()